# relay_write

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): the report and the figure of `relay_report`'s summaries, and synthetic rows with known effects. `writeC1b` (`report.md`: Gate C1b, both parts, with its verdict and what it decides for A1; H23-3 to H23-8; per arm honest pooled share beside contamination; the bridge, relay and core measures; every `C1B_PLAN` item), `figC1b` (`c1b.png`: effectiveness and safety per arm, side by side) and `syntheticRows23` (every planned cell), for `test_gate_c1b`, `test_write_c1b` and the quick demo.

In [ ]:
#| default_exp relay_write

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json

import numpy as np

from explRL.bridge_write import _HEAD, _ci, _g, _row, _slots
from explRL.colony_report import NAN
from explRL.relay_grid import ARMS23, N20
from explRL.relay_plan import C1B_PLAN
from explRL.relay_report import planned23


def writeC1b(path, S, meta, plan=C1B_PLAN):
    """report.md (FR-D4): Gate C1b (both parts, part (i) on welfareHonest too), its verdict and what it decides; H23-3 to H23-8;
    per arm honest pooled share and contamination side by side; the columns and core measures; every plan item."""
    G, H, A = S['gate'], S['hyp'], S['arms']
    L = ['# Phase 23 (C1b): bridges that carry evidence (exploratory)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}, boards {S['boards']}", '',
         '## Gate C1b (64 agents, independent liars, bridgeFix - bridge1, seeds 1800-1807 paired)', ''] + _HEAD
    ii = G['ii']
    L += [_row('(i) effective: honestPooled', G['i'], f"passes: {G['i']['passes']}"),
          _row('(i) robustness: welfareHonest', G['welfareHonest'], f"same sign: {G['robust']}"),
          _row('(ii) safe: contamination (p for < 0: safer)', ii,
               f"holds: {ii['holds']} (95th percentile of paired bootstrap means {_g(ii['upper'])} against {ii['margin']})")]
    L += ['', f"**Gate C1b: {G['verdict']}.** {G['decides']}.", '', '## Secondary (seeds 1800-1804; exploratory, uncorrected)', ''] + _HEAD
    h3, h5 = H['H23_3'], H['H23_5']
    L += [_row('H23-3 margin: (bridgeFix - coupled) - 0.5 (squads - coupled)', h3,
               f"win-back {_g(h3['winBack'])} {_ci(h3['winBackCi'])}; supported: {h3['supported']}; truthConfirmable "
               f"{_g(h3['truthConfirmable'][0])}"),
          _row('coupling cost: squads - coupled', h3['couplingCost']), _row('bridge gain: bridgeFix - coupled', h3['bridgeGain']),
          _row('H23-4 filter gain, bridgeFix boards', H['H23_4'], f"supported: {H['H23_4']['supported']}"),
          _row('H23-5 evidence (mean of its two cell differences)', h5['evidence']), _row('H23-5 dedicated role', h5['role']),
          _row('H23-5 interaction: (bridgeFix - bridgeRole) - (relayEvid - bridge1)', h5['interaction'])]
    L += [_row(f'H23-6 {c}', t) for c, t in H['H23_6'].items()]
    L += [_row(f'H23-7 clean: {c}, {m} (p for < 0: a cost)', t) for c, dd in H['H23_7']['contrasts'].items() for m, t in dd.items()]
    L += [_row(f'H23-8 coupledContamination: {a} - coupled (p for < 0)', t) for a, t in H['H23_8']['againstCoupled'].items()]
    L += ['', 'H23-5 cells (honestPooled, 1800-1804): ' + '; '.join(f'{a} {_g(v[0])} {_ci(v[1:])}' for a, v in h5['cells'].items()),
          'H23-7 clean boards: ' + '; '.join(f"{a} honestPooled {_g(v['honestPooled'])}" for a, v in H['H23_7']['perArm'].items()),
          'H23-8 coupledContamination per arm: ' + '; '.join(f'{a} {_g(v[0])} {_ci(v[1:])}' for a, v in H['H23_8']['perArm'].items()), '',
          '## Effective and safe, per arm (independent liars)', '',
          '| arm | boards | honestPooled | 95% interval | contamination | 95% interval |', '|---|---|---|---|---|---|']
    L += [f"| {a} | {v['boards']} | {_g(v['honestPooled'][0])} | {_ci(v['honestPooled'][1:])} | {_g(v['contamination'][0])} | "
          f"{_ci(v['contamination'][1:])} |" for a, v in A.items()]
    for title, k in (('Core measures, boards with liars', 'core'), ('Core measures, clean boards', 'coreClean'),
                     ('Phase 18, bridge and relay columns (means over boards with liars)', 'columns')):
        ks = list(next(v[k] for v in A.values() if v[k]))
        L += ['', f'## {title}', '', '| arm | ' + ' | '.join(ks) + ' |', '|---|' + '---|' * len(ks)]
        L += [f'| {a} | ' + ' | '.join(_g(v[k][c]) for c in ks) + ' |' for a, v in A.items() if v[k]]
    L += ['', '## C1B_PLAN', ''] + [f'- {k}: {json.dumps(v, default=str)}' for k, v in plan.items()]
    open(path, 'w').write('\n'.join(L) + '\n')


def figC1b(out, S):
    "c1b.png (FR-D4): per arm with liars, honest pooled share (left) and contamination with bridge1 + the margin (right)."
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    A, G = S['arms'], S['gate']
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, m, title in ((axes[0], 'honestPooled', 'Effective: honest pooled share'), (axes[1], 'contamination', 'Safe: contamination')):
        for x, (arm, v) in enumerate(A.items()):
            e, lo, hi = v[m]
            if np.isfinite(e):
                ax.errorbar([x], [e], yerr=[[e - lo], [hi - e]], fmt='o', capsize=3, ms=7,
                            color='#1f6391' if arm in ('relayEvid', 'bridgeRole', 'bridgeFix', 'mirrorWindow') else '#555555')
        ax.set_xticks(range(len(A)), list(A), rotation=30, ha='right', fontsize=8)
        ax.set(ylabel=m, title=title)
        ax.spines[['top', 'right']].set_visible(False)
    ref = A['bridge1']['contamination'][0]
    axes[1].axhline(ref + G['ii']['margin'], color='#c0392b', lw=1, ls='--', label='bridge1 + 0.05: the bar of part (ii)')
    axes[1].legend(fontsize=7, frameon=False)
    fig.suptitle(f"64 agents with liars; Gate C1b {G['verdict']}", fontsize=10)
    fig.tight_layout()
    fig.savefig(f'{out}/c1b.png', dpi=150, metadata={'Software': None})
    plt.close(fig)

In [ ]:
#| export
def syntheticRows23(squads=0.24, cost=0.09, bridge1=0.01, ev=0.025, role=0.025, inter=0.0, window=0.08, safety=0.0, safetyNoise=0.0,
                    clean=0.40, noise=0.02, seed=0, plan=C1B_PLAN):
    """Rows of every planned C1b cell (N 64; planned23) with known effects. honestPooled (welfareHonest 1.3 times it): squads;
    coupled = squads - cost; bridge1 = coupled + bridge1; relayEvid = bridge1 + ev; bridgeRole = bridge1 + role; bridgeFix =
    bridge1 + ev + role + inter; mirrorWindow = coupled + window. Contamination 0.20 on the bridge arms, bridgeFix 0.20 + safety
    (+ a per-seed draw of sd safetyNoise), mirrorWindow 0.22, coupled 0.18, squads 0.15. One noise draw per seed is shared
    by every arm, so paired differences are exact. Clean boards: clean, minus 0.03 (coupled), 0.058 (bridge1), 0.02 (bridgeFix),
    0.01 (mirrorWindow). Relay columns: filterGain 0.2, arrivalConfirmed 0.6 on evidence arms (0.3 otherwise), truthConfirmable
    0.52, liarConfirmed 0.018; coupledContamination 0.05 (coupled), 0.06 elsewhere."""
    rng, rows = np.random.default_rng(seed), []
    want = sorted(planned23(plan))
    e = {s: rng.normal(0, noise) for s in sorted({s for _, s in want})}
    es = {s: rng.normal(0, safetyNoise) if safetyNoise else 0.0 for s in e}
    hp = dict(squads=squads, coupled=squads - cost)
    hp.update(bridge1=hp['coupled'] + bridge1, mirrorWindow=hp['coupled'] + window)
    hp.update(relayEvid=hp['bridge1'] + ev, bridgeRole=hp['bridge1'] + role, bridgeFix=hp['bridge1'] + ev + role + inter)
    hc = dict(squads=clean, coupled=clean - 0.03, bridge1=clean - 0.058, bridgeFix=clean - 0.02, mirrorWindow=clean - 0.01)
    con = dict(squads=0.15, coupled=0.18, mirrorWindow=0.22)
    for lv, s in want:
        a, f = lv.split('|')
        w, nl = ARMS23[a], 0 if f == 'clean' else 16
        x, br, cp = (hc[a] if nl == 0 else hp[a]) + e[s], w.get('bridges'), w.get('couple')
        c = con.get(a, 0.20) + (safety + es[s] if a == 'bridgeFix' else 0.0)
        rel = dict(filterGain=0.2 if br else NAN, arrivalConfirmed=(0.6 if w.get('evidence') else 0.3) if br else NAN,
                   reRelays=1.0 if br else NAN, roleBreaks=0.0 if w.get('bridgeRole') else NAN,
                   coupledContamination=(0.05 if a == 'coupled' else 0.06) if cp else NAN,
                   truthConfirmable=0.52 if cp else NAN, liarConfirmed=0.018 if cp else NAN)
        rows.append(dict(level=lv, seed=s, N=N20, nLiar=nl, tEnd=40.0, honestPooled=x, welfareHonest=1.3 * x, welfare=x,
                         contamination=c, falseRelayShare=0.3 if br else NAN, crossContamination=0.05 if cp else NAN,
                         bridgeRelays=30.0 if br else 0.0, relaysPerBridgeTurn=0.1 if br else NAN, mirrorPosts=200.0 if cp else 0.0,
                         inTokPerCall=800.0, honestTurns=40.0 * (N20 - nl), misinformedFirst=6.0 if nl else 0.0,
                         liarFirst=4.0 if nl else 0.0, uncorrected=2.0 if nl else 0.0, liarPosts=20.0 if nl else 0.0,
                         contests=10.0 if nl else 0.0, guardLies=8.0 if nl else 0.0, contestLags=json.dumps([1.0, 3.0] if nl else []),
                         wrongHonest=6.0, sameWrong=2.0, sharePerHeld=0.5, inputTokens=1.5e6, outputTokens=3e5, solved=30.0,
                         **rel, **_slots(nl)))
    return rows

In [ ]:
# test_write_c1b (FR-D4): the report states the gate's verdict and what it decides, both parts, H23-3 to H23-8 and per arm
# honest pooled share beside contamination; the figure is written; a safety failure is reported as such
import tempfile
from pathlib import Path
from explRL.relay_report import c1bSummaries
d = Path(tempfile.mkdtemp())
for kw, verdict in ((dict(), 'passes'), (dict(safety=0.08), 'noBridges')):
    S = c1bSummaries(syntheticRows23(**kw), nboot=200)
    writeC1b(d / 'report.md', S, dict(mode='test', planHash='h', commit='c'))
    figC1b(d, S)
    t = (d / 'report.md').read_text()
    assert f'**Gate C1b: {verdict}.**' in t and S['gate']['decides'] in t
    for s in ('(i) effective', '(ii) safe', 'H23-3', 'H23-4', 'H23-5', 'H23-6', 'H23-7', 'H23-8', '| arm | boards | honestPooled |',
              'contamination | 95% interval', 'arrivalConfirmed', 'truthConfirmable', '## C1B_PLAN', '- cap: 250.0'):
        assert s in t, s
    assert (d / 'c1b.png').stat().st_size > 10_000
print(t[:1800])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()